# **Review and Correction of the Dataset**

The main goal of this notebook is *review* and *correction* of the output from *pre-labeling* pipeline made in the [notebook 04_annotate_DNIs_ds_v1.0.0.ipynb](./04_annotate_DNIs_ds_v1.0.0.ipynb).

To do this task, we will load our *pre-labeling dataset* to **FiftyOne** to show the OOB *(Oriented Bounding Boxes)* generated by the previous pipeline. Then load our samples to **CVAT** to check and correct the labels that were wrong.

## **Expected Input:**
* First version of *pre-labeling dataset* from the previous step.

## **Expected Output:**
* Checked dataset ready to make the splits for the *training* process.

## **Libraries, Environment Variables and Settings**

In [20]:
import os
import getpass
if not os.environ.get('FIFTYONE_CVAT_USERNAME'):
    os.environ['FIFTYONE_CVAT_USERNAME'] = input("CVAT Username: ")
if not os.environ.get('FIFTYONE_CVAT_PASSWORD'):
    os.environ['FIFTYONE_CVAT_PASSWORD'] = getpass.getpass("CVAT Password: ")
if not os.environ.get('FIFTYONE_CVAT_URL'):
    os.environ['FIFTYONE_CVAT_URL'] = "http://localhost:8080"
if not os.environ.get("BASE_PROJECT_DIR"):
    os.environ["BASE_PROJECT_DIR"] = input("Enter the base project directory path: ")
import fiftyone as fo
import glob

DS_DIR = os.getenv("BASE_PROJECT_DIR") + "/data/prc/input/"
DS_EXPORT_DIR = os.getenv("BASE_PROJECT_DIR") + "/data/ds/ds_arg_ID_card_det_v1"
DS_NAME = "verify_dni_labels"
CLASSES = ["front", "back"]
ANNO_KEY = "CVAT_RUN_00"
PROJECT_NAME = "argentinian-id-card-detection"

print(f"FiftyOne version: {fo.__version__}")

FiftyOne version: 1.20.1


Load Dataset to **FiftyOne**

*Notes:* to open the **FiftyOne** Dashboard execute in a terminal the next command:
```
fiftyone app launch
```

In [2]:
if fo.dataset_exists(DS_NAME):
    print(f"Dataset '{DS_NAME}' already exists. Loading the existing dataset...")
    dataset = fo.load_dataset(DS_NAME)
    # fo.delete_dataset(DS_NAME)
else:
    dataset = fo.Dataset(DS_NAME)

    for image in glob.glob(DS_DIR+"images/*"):
        label_path = DS_DIR + "labels/" + os.path.splitext(os.path.basename(image))[0] + ".txt"
        if not os.path.exists(label_path):
            raise FileNotFoundError(f"Label file not found for image: {image}")
        polygons = []
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                coords = [float(val) for val in parts[1:]]

                points = [
                    [coords[0], coords[1]],
                    [coords[2], coords[3]],
                    [coords[4], coords[5]],
                    [coords[6], coords[7]],
                ]

                polygons.append(
                    fo.Polyline(label=CLASSES[cls_id],
                                points=[points],
                                closed=True,
                                # filled=True
                    )
                )

        sample = fo.Sample(filepath=image)
        sample["ground_truth"] = fo.Polylines(polylines=polygons)
        dataset.add_sample(sample)

dataset.persistent = True
dataset

Dataset 'verify_dni_labels' already exists. Loading the existing dataset...


Name:        verify_dni_labels
Media type:  image
Num samples: 880
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Polylines)

Load the samples to **CVAT**

In [ ]:
view = dataset

view.annotate(
    ANNO_KEY,
    project_name=PROJECT_NAME,
    label_field="ground_truth",
    label_type="polylines",
    classes=CLASSES,
    image_quality=100,
    overwrite=True

)

view.get_annotation_info(ANNO_KEY)

Computing metadata...
 100% |█████████████████| 880/880 [95.3ms elapsed, 0s remaining, 9.2K samples/s] 
Uploading samples to CVAT...


{
    "key": "CVAT_RUN_00",
    "version": "1.20.1",
    "timestamp": "2026-08-12T02:18:43.186179",
    "config": {
        "cls": "fiftyone.utils.cvat.CVATBackendConfig",
        "type": "annotation",
        "method": "cvat",
        "overwrite": true,
        "name": "cvat",
        "label_schema": {
            "ground_truth": {
                "type": "polylines",
                "classes": [
                    {
                        "classes": [
                            "front"
                        ],
                        "attributes": {}
                    },
                    {
                        "classes": [
                            "back"
                        ],
                        "attributes": {}
                    }
                ],
                "attributes": {},
                "existing_field": true,
                "allow_additions": true,
                "allow_deletions": true,
                "allow_label_edits": true,
           

* Load the corrected annotations to our dataset and export it

In [19]:
dataset.load_annotations(ANNO_KEY)

Download complete
Loading labels for field 'ground_truth'...
 100% |█████████████████| 880/880 [693.4ms elapsed, 0s remaining, 1.3K samples/s]       


In [21]:
dataset.export(
    export_dir=DS_EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    split="train"
)

 100% |█████████████████| 880/880 [720.0ms elapsed, 0s remaining, 1.2K samples/s]       


## **Conclusions**

The results of the pre-annotation pipeline were excellent. The detection of the OBBs was excellent and highly accurate; only some label corrections were needed, never any OBB corrections.

Therefore, we were able to review and correct our dataset of 902 samples in less than an hour, since only labels needed correction. The model chosen (SAM3) for the consolidation process significantly altered the text labels, but at no point did we need to correct the bounding boxes.

To divide our dataset into *train*, *validation*, and *test* sets for the training process, it was decided to inspect the samples and manually select the samples for the *validation* and *test* sets.
It should be noted that all synthetic examples will be used in the *training* set, as well as approximately 60% of the documents obtained from the internet (public Google Images) and from family or friends who agreed to provide their identity documents for this academic and research project.